# 05b — RAG, Approach B: link everything except the test set

*The second route you asked for.* One dense index over the **entire corpus** (`rag_corpus.csv` = all loans except the 1000-row test batch). For each test loan, retrieve the **top-`K_PRECEDENTS` nearest precedents** by cosine similarity of their content embeddings and inject them as labelled evidence — a classic kNN / retrieval-augmented few-shot setup.

No Semantic IDs, no staging: the whole corpus is a single flat index. This is the simpler counterpart to Approach A, and the contrast between them is the point.

Compared against a **no-RAG LLM** baseline and the **XGBoost** baseline on the same 1000 loans.

In [ ]:
import sys; sys.path.insert(0, '..')
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from llm_utils import (
    run_llm_experiment, run_ml_on_sample, evaluate_predictions,
    build_system_prompt, build_user_prompt, load_all_api_keys, RESULTS_DIR,
)
from sample_generation import get_test_batch
import rag_utils as R

# ── Config — change these to switch provider / backend ──────────────
API_PROVIDER      = 'nvidia'                       # only key present in .env
MODEL_NAME        = 'meta/llama-3.3-70b-instruct'
MODEL_LABEL       = 'Llama-3.3-70B'
EMBEDDING_BACKEND = 'auto'                         # sentence-transformers if installed, else TF-IDF
K_PRECEDENTS      = 8                              # precedents injected per loan
N_STAGE1          = 50                             # stage-1 pool (Approach A)
N_TEST            = None                           # e.g. 50 for a quick smoke run; None = all 1000
FORCE_RERUN       = False                          # True ignores the call cache
NOTEBOOK_ID       = '05b_RAG_FullCorpus_Retrieval.ipynb'
WITH_LOGPROBS     = API_PROVIDER in ('openai', 'nvidia', 'gemini')

API_KEYS    = load_all_api_keys(API_PROVIDER)
MAX_WORKERS = max(1, len(API_KEYS) * 4)  # lower this if a single key hits NVIDIA 429s
print(f'{len(API_KEYS)} API key(s) for {API_PROVIDER}; MAX_WORKERS={MAX_WORKERS}')

In [ ]:
# One-time: install the embedding backend if needed (pulls torch).
# Uncomment, run once, then restart the kernel.
# %pip install sentence-transformers
try:
    import sentence_transformers  # noqa: F401
    print('sentence-transformers available')
except Exception:
    print('sentence-transformers NOT installed — Embedder will fall back to TF-IDF.\n'
          'Install it (cell above) for the paper-faithful content embeddings.')

## 1. Corpus + test set (leakage-checked)

In [ ]:
corpus = R.build_rag_corpus()          # all loans EXCEPT the eval batches
test   = get_test_batch()              # the 1000-row FINAL test set
if N_TEST:
    test = test.iloc[:N_TEST].reset_index(drop=True)
R.assert_no_leakage(corpus, test)
print(f'Corpus={len(corpus)}  Test={len(test)}  (no overlap)')

## 2. Embed the full corpus and the test loans
A single content-embedding space over all 30 LLM features — the flat index every test loan is matched against.

In [ ]:
emb = R.Embedder(backend=EMBEDDING_BACKEND)
corpus_emb = emb.fit_transform(R.serialize_frame(corpus))
test_emb   = emb.transform(R.serialize_frame(test))
print('embedding backend:', emb._resolved, '| corpus', corpus_emb.shape, '| test', test_emb.shape)

## 3. Top-k retrieval → precedent context
For each test loan, the `K_PRECEDENTS` nearest labelled precedents over the whole corpus.

In [ ]:
blocks = []
for i in range(len(test)):
    idx, sims = R.cosine_topk(test_emb[i], corpus_emb, K_PRECEDENTS)
    blocks.append(R.build_precedent_block(corpus.iloc[idx], sims=sims))
print(f'Built {len(blocks)} precedent contexts.')
print('\n--- example precedent block (test loan 0) ---\n')
print(blocks[0][:900])

## 4. Run the RAG LLM
⚠️ One API call per loan — set `N_TEST` small first.

In [ ]:
test_rag, rag_prompt_fn = R.make_rag_prompt_fn(test, blocks)
rag_system_prompt = R.build_rag_system_prompt()

rag = run_llm_experiment(
    test_rag, api_provider=API_PROVIDER, model_name=MODEL_NAME,
    api_keys=API_KEYS, max_workers=MAX_WORKERS, use_cache=not FORCE_RERUN,
    label=f'{MODEL_LABEL} | RAG-B full-corpus kNN', include_desc=False,
    with_logprobs=WITH_LOGPROBS, system_prompt=rag_system_prompt,
    user_prompt_fn=rag_prompt_fn, notebook_id=NOTEBOOK_ID,
)
print('RAG metrics:', rag['metrics'])

## 5. Baselines — no-RAG LLM and XGBoost (same 1000 loans)

In [ ]:
base = run_llm_experiment(
    test, api_provider=API_PROVIDER, model_name=MODEL_NAME,
    api_keys=API_KEYS, max_workers=MAX_WORKERS, use_cache=not FORCE_RERUN,
    label=f'{MODEL_LABEL} | no-RAG', include_desc=False,
    with_logprobs=WITH_LOGPROBS, notebook_id=NOTEBOOK_ID,
)

xgb_probs, xgb_preds = run_ml_on_sample(test)
xgb_metrics = evaluate_predictions(test['loan_status'].values, xgb_preds,
                                   label='XGBoost', probabilities=xgb_probs)

## 6. Compare + save results

In [ ]:
def _row(name, m):
    return {'model': name, **{k: m.get(k) for k in
            ['accuracy', 'precision_charged_off', 'recall_charged_off', 'f1_charged_off', 'auc']}}

summary = pd.DataFrame([
    _row('RAG-B (full-corpus kNN)', rag['metrics']),
    _row('LLM no-RAG', base['metrics']),
    _row('XGBoost', xgb_metrics),
])
display(summary)
summary.to_csv(f'{RESULTS_DIR}/05b_summary.csv', index=False)

preds = pd.DataFrame({
    'loan_index': range(len(test)),
    'actual':     test['loan_status'].values,
    'rag_pred':   rag['predictions'],
    'rag_prob':   rag['probabilities'],
    'norag_pred': base['predictions'],
    'xgb_pred':   xgb_preds,
    'xgb_prob':   xgb_probs,
})
preds.to_csv(f'{RESULTS_DIR}/05b_predictions.csv', index=False)
print(f'Saved -> {RESULTS_DIR}/05b_summary.csv  and  05b_predictions.csv')

In [ ]:
ax = summary.set_index('model')[['accuracy', 'f1_charged_off']].plot.bar(rot=15, figsize=(8, 4))
ax.set_title('Approach B (full-corpus kNN RAG) vs baselines')
ax.set_ylabel('score'); plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/05b_comparison.png', dpi=150, bbox_inches='tight'); plt.show()